# 🧬 MIGN-XAI: Full Pipeline Notebook

**Multimodal Interaction Graph-Omics Network with Explainability for Cancer Drug Response Prediction**

This notebook executes the complete research pipeline end-to-end:

| Phase | Steps | Key Features |
|-------|-------|-------------|
| **Setup** | 1-2 | Install deps, configure paths |
| **Data** | 3-7 | Download, preprocess GDSC + omics + drug graphs, verify splits with leakage checks |
| **Training** | 8 | Cell-blind split, Huber loss + LDS, GPU monitoring |
| **Evaluation** | 9-10 | All splits + failure analysis + MC Dropout uncertainty |
| **Experiments** | 11-14 | Cross-dataset (GDSC→CCLE), ablation (Δ analysis), baselines (Wilcoxon tests) |
| **XAI** | 15-17 | SHAP (+ stability), attention viz, biological validation |
| **Results** | 18-19 | Display all plots, integration test verification |

---
## 1. Install Dependencies

In [ ]:
# Run in Google Colab or local Python 3.10+ environment
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q torch-geometric
!pip install -q rdkit-pypi pubchempy shap gseapy openpyxl tqdm seaborn
!pip install -q scipy scikit-learn optuna pandas numpy matplotlib

## 2. Configure Project Paths & Seed Reproducibility

In [ ]:
import os, sys

# If running in Colab, clone or mount the project
PROJECT_ROOT = os.path.abspath('.')
if 'cancer_drp_project' in os.listdir(PROJECT_ROOT):
    PROJECT_ROOT = os.path.join(PROJECT_ROOT, 'cancer_drp_project')
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

print(f'Project root: {PROJECT_ROOT}')
print(f'Contents: {os.listdir(".")}')

# Lock ALL random seeds for full reproducibility
from utils.helpers import set_seed
set_seed(42)
print('\nSeed locked: random, numpy, torch, cudnn, PYTHONHASHSEED, deterministic_algorithms')

---
## 3. Download Data

In [ ]:
from data.download_data import download_all
download_all()

## 4. Preprocess GDSC IC50 Data

In [ ]:
from data.preprocess_gdsc import preprocess_gdsc
preprocess_gdsc()

## 5. Preprocess Omics (Gene Expression + Mutations + Pathways)

In [ ]:
from data.preprocess_omics import preprocess_omics
preprocess_omics()

## 6. Build Drug Molecular Graphs

In [ ]:
from data.drug_graph import process_drug_graphs
drug_graphs = process_drug_graphs()

## 7. Verify Dataset Splits + Data Leakage Checks

Each split strategy now runs **automatic leakage verification**:
- `cell_blind`: AssertionError if any cell line appears in both train and test
- `drug_blind`: AssertionError if any drug appears in both train and test
- `random`: Reports overlap percentage (expected — this is why random splits are weak)

In [ ]:
from data.dataset import load_all_data

splitter = load_all_data('gdsc1')

for strategy in ['random', 'cell_blind', 'drug_blind']:
    print(f'\n{"="*60}')
    print(f'  Split Strategy: {strategy}')
    print(f'{"="*60}')
    train_ds, val_ds, test_ds = splitter.get_split(strategy)
    sample = train_ds[0]
    print(f'  Sample: atoms={sample.x.shape}, cell={sample.cell.shape}, y={sample.y.item():.4f}')

---
## 8. Train MIGN-XAI (Cell-Blind Split)

Training features:
- **Huber Loss** (robust to IC50 outliers)
- **Label Distribution Smoothing** (oversamples rare IC50 values)
- **L1 regularisation** on cell encoder layer 1 (sparse gene selection)
- **Cosine annealing** LR schedule
- **GPU memory monitoring** + training time tracking
- **Early stopping** on validation Pearson r (patience=20)

In [ ]:
from experiments.train import train

test_metrics = train(
    split_strategy='cell_blind',
    n_epochs=200,
    lr=3e-4,
    use_lds=True,
    seed=42,
    tag='v1'
)
print('\nFinal test metrics:', test_metrics)

---
## 9. Evaluate Across All Splits

Includes:
- Per-split metrics (Pearson r, Spearman ρ, RMSE, AUROC)
- Per-drug breakdown
- **Failure case analysis** (top 10 worst predictions + error patterns)
- **MC Dropout uncertainty estimation** (on cell_blind split)

In [ ]:
from experiments.evaluate import evaluate

all_results = evaluate(
    splits=['random', 'cell_blind', 'drug_blind'],
    checkpoint_path=None  # uses best_model.pt
)

# Print uncertainty results if available
if 'cell_blind' in all_results and 'uncertainty' in all_results['cell_blind']:
    u = all_results['cell_blind']['uncertainty']
    print(f'\nUncertainty Summary:')
    print(f'  Mean sigma: {u["mean_uncertainty"]:.4f}')
    print(f'  Error-uncertainty correlation: {u["error_uncertainty_spearman"]:.4f}')
    print(f'  Coverage +/-2sigma: {u["coverage_2sigma_pct"]:.1f}%')

## 10. Manual MC Dropout Uncertainty Demo

MC Dropout runs T stochastic forward passes with dropout enabled.
Output = mean prediction ± uncertainty (std).

In [ ]:
import torch
import numpy as np
from torch_geometric.loader import DataLoader as GeoLoader
from models.mign_xai import build_model
from config import CKPT_DIR, PROC_DIR, EVAL_BATCH_SIZE
import pandas as pd

# Load model
cell_features = pd.read_csv(PROC_DIR / 'cell_features.csv', index_col=0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model(cell_in=cell_features.shape[1], device=device)

ckpt = torch.load(CKPT_DIR / 'best_model.pt', map_location=device, weights_only=False)
model.load_state_dict(ckpt['model'])

# Get a small test batch
splitter = load_all_data('gdsc1')
_, _, test_ds = splitter.get_split('cell_blind')
test_loader = GeoLoader(test_ds, batch_size=16, shuffle=False)
batch = next(iter(test_loader)).to(device)

# MC Dropout: 30 stochastic forward passes
mean, std, all_preds = model.mc_dropout_predict(batch, n_forward=30)

print('MC Dropout Results (first 10 samples):')
print(f'{"True":>8} {"Pred":>8} {"Uncertainty":>12} {"In 2sigma?":>12}')
for i in range(min(10, len(mean))):
    true_val = batch.y[i].item()
    in_range = abs(true_val - mean[i]) <= 2 * std[i]
    print(f'{true_val:>8.3f} {mean[i]:>8.3f} {std[i]:>10.4f}   {"YES" if in_range else "NO":>8}')

---
## 11. Cross-Dataset Generalisation (GDSC → CCLE)

**This is the real generalisation test.**

| Level | Test | Difficulty |
|-------|------|------------|
| Sanity | GDSC1 → GDSC2 | Easy (same lab, same protocol) |
| **Real** | **GDSC → CCLE** | **Hard (different lab, different assay, different sensitivity measure)** |

Uses Spearman rank correlation (not Pearson) because GDSC measures ln(IC50) and CCLE measures AUC.

In [ ]:
from experiments.cross_dataset import run_cross_dataset

cross_results = run_cross_dataset()

## 12. Ablation Study (7 Variants + Δ Analysis)

Tests each architectural component by removing it and measuring performance drop.

Reports: ΔPearson r, ΔRMSE, component contribution analysis, and verifies full model is best.

In [ ]:
from experiments.ablation import run_ablation

run_ablation()

## 13. Baseline Models + Wilcoxon Significance Tests

All baselines use the **exact same cell-blind split** (verified by MD5 hash).

Improvements are tested with the **Wilcoxon signed-rank test** (paired, non-parametric).

| Baseline | Drug Features | Fusion |
|----------|--------------|--------|
| SVM | ECFP fingerprints | Concat |
| Random Forest | ECFP fingerprints | Concat |
| GraphDRP-GIN | GIN graph | Concat |
| **MIGN-XAI** | **GIN + edges** | **Cross-attention** |

In [ ]:
from experiments.baselines import run_baselines

run_baselines()

## 14. Manual Wilcoxon Significance Test

Compare any two models using the Wilcoxon signed-rank test on per-sample squared errors.

In [ ]:
from utils.metrics import statistical_significance_test

# Load saved predictions
mign_preds = pd.read_csv('results/eval_cell_blind_predictions.csv')
y_true = mign_preds['y_true'].values
y_pred_mign = mign_preds['y_pred'].values

# Example: compare with a dummy worse model (add noise)
np.random.seed(42)
y_pred_dummy = y_pred_mign + np.random.randn(len(y_pred_mign)) * 0.5

result = statistical_significance_test(
    y_true, y_pred_dummy, y_pred_mign,
    'Noisy Model', 'MIGN-XAI'
)
print(f'\nVerdict: {result["better_model"]} wins with p={result["p_value"]:.2e} ({result["significance"]})')

---
## 15. SHAP Gene Attribution + Stability Verification

KernelSHAP with:
- Drug-specific attributions (fix drug embedding, vary cell features)
- **Stability check**: 3 runs with different seeds, reports Top-10 Jaccard + Rank Spearman
- **Cross-drug consistency**: Do EGFR drugs (Erlotinib, Gefitinib) identify similar genes?

In [ ]:
from results.xai.shap_analysis import run_shap_analysis

run_shap_analysis()

## 16. Attention Weight Visualisation

In [ ]:
from results.xai.attention_viz import run_attention_viz

run_attention_viz()

## 17. Biological Validation (Precision@K + Fisher's Test)

In [ ]:
from results.xai.biological_val import run_biological_validation

val_df = run_biological_validation()
if val_df is not None:
    display(val_df)

---
## 18. Display All Results

In [ ]:
from IPython.display import Image, display
from pathlib import Path
from config import RESULT_DIR

plots_dir = RESULT_DIR / 'plots'
xai_dir = RESULT_DIR / 'xai'

# Scatter plots (one per split)
print('=' * 60)
print('  SCATTER PLOTS')
print('=' * 60)
for f in sorted(plots_dir.glob('scatter_*.png')):
    print(f'\n{f.stem}:')
    display(Image(filename=str(f), width=500))

# SHAP bar charts (one per drug)
print('\n' + '=' * 60)
print('  SHAP GENE ATTRIBUTIONS')
print('=' * 60)
for f in sorted(xai_dir.glob('shap_bar_*.png')):
    print(f'\n{f.stem}:')
    display(Image(filename=str(f), width=600))

# Attention heatmap
attn_heatmap = xai_dir / 'attention_heatmap_drugs.png'
if attn_heatmap.exists():
    print('\nAttention Heatmap:')
    display(Image(filename=str(attn_heatmap), width=600))

# Biological validation
for name in ['precision_at_10.png', 'biological_validation_precision.png']:
    p = xai_dir / name
    if p.exists():
        print(f'\nBiological Validation ({name}):')
        display(Image(filename=str(p), width=700))

## 19. Integration Test Verification

Run 7 automated tests to verify the entire pipeline is working correctly.

In [ ]:
import torch
import numpy as np
from torch_geometric.data import Data, Batch
from models.mign_xai import MIGN_XAI
from utils.helpers import set_seed
from utils.metrics import compute_metrics, statistical_significance_test
from utils.lds import get_lds_weights

model = MIGN_XAI(node_in=45, edge_in=10, cell_in=1026)
samples = [
    Data(x=torch.randn(10, 45),
         edge_index=torch.tensor([[0,1,2,3],[1,2,3,4]], dtype=torch.long),
         edge_attr=torch.randn(4, 10),
         cell=torch.randn(1, 1026),
         y=torch.tensor([float(i)]))
    for i in range(4)
]
batch = Batch.from_data_list(samples)

# Test 1: Forward pass
model.eval()
y_hat, attn = model(batch)
assert y_hat.shape == torch.Size([4])
print(f'Test 1 - Forward pass: {y_hat.shape} OK')

# Test 2: MC Dropout uncertainty
mean, std, _ = model.mc_dropout_predict(batch, n_forward=10)
assert mean.shape == (4,) and std.shape == (4,)
print(f'Test 2 - MC Dropout: uncertainty range [{std.min():.4f}, {std.max():.4f}] OK')

# Test 3: Seed reproducibility
set_seed(42); a = torch.randn(5)
set_seed(42); b = torch.randn(5)
assert torch.equal(a, b)
print(f'Test 3 - Reproducibility: OK')

# Test 4: Wilcoxon test
y_t = np.random.randn(100)
r = statistical_significance_test(y_t, y_t + np.random.randn(100)*2, y_t + np.random.randn(100)*0.1, 'Bad', 'Good')
assert r['better_model'] == 'Good'
print(f'Test 4 - Wilcoxon: p={r["p_value"]:.2e} OK')

# Test 5: Metrics
m = compute_metrics(np.array([1,2,3,4]), np.array([1.1,2.2,2.8,4.3]))
assert m['pearson_r'] > 0.95
print(f'Test 5 - Metrics: r={m["pearson_r"]:.3f} OK')

# Test 6: LDS
w = get_lds_weights(np.random.randn(200))
assert w.min() >= 1.0
print(f'Test 6 - LDS: range [{w.min():.1f}, {w.max():.1f}] OK')

# Test 7: SHAP path
model.eval()
z = model.get_drug_embedding(Batch.from_data_list([samples[0]]))
preds = model.predict_from_embeddings(z, np.random.randn(5, 1026).astype(np.float32))
assert preds.shape == (5,)
print(f'Test 7 - SHAP path: OK')

print('\n' + '='*50)
print('  ALL 7 INTEGRATION TESTS PASSED')
print('='*50)

---
## ✅ Pipeline Complete

### All outputs saved to `results/`:

| File | Contents |
|------|----------|
| `evaluation_results.json` | Metrics for all splits + uncertainty |
| `eval_cell_blind_predictions.csv` | Per-sample predictions |
| `eval_cell_blind_failures.csv` | Top worst predictions + error patterns |
| `ablation_results.json` | 7-variant ablation + Δ analysis |
| `baseline_results.json` | Baselines + Wilcoxon significance |
| `cross_dataset_results.json` | GDSC→CCLE + GDSC1→GDSC2 |
| `xai/shap_*.csv` | Per-drug SHAP attributions |
| `xai/shap_stability.json` | Multi-run consistency scores |
| `xai/biological_validation.csv` | Precision@K + Fisher's p-values |
| `plots/scatter_*.png` | Prediction scatter plots |
| `checkpoints/best_model.pt` | Trained model weights |

### Key Research Features:
- ✅ Data leakage verification (AssertionError on leak)
- ✅ MC Dropout uncertainty estimation
- ✅ Wilcoxon signed-rank significance testing
- ✅ Failure case analysis (worst predictions + patterns)
- ✅ SHAP stability verification (Jaccard + Spearman across runs)
- ✅ Cross-dataset generalisation (GDSC → CCLE)
- ✅ Full reproducibility (7 random seed locks)
- ✅ Ablation with Δ contribution analysis